In [1]:
from transformers import CanineModel, CanineTokenizer
from classes.style_encoder import StyleEncoder
from classes.conversational_dataset import ConversationDataset
from torch.utils.data import DataLoader
from torch.nn.functional import normalize
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import emoji
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import string
import os
import json
import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

Skipping import of cpp extensions due to incompatible torch version 2.7.1+cu118 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1226 20:32:19.184000 16536 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# =============================
# SANITY CHECK: Embedding Norms
# =============================
# Purpose: Ensure embeddings are not collapsed and L2 normalization works.
# - mean/std check tells you if embeddings vary
# - norm check ensures embeddings are unit vectors

def evaluate_embedding_sanity(model, dataloader, device):
    model.eval()
    all_embeddings = []

    with torch.no_grad():
        for batch in dataloader:
            z = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device)
            )
            all_embeddings.append(z.cpu())

    embeddings = torch.cat(all_embeddings)

    return {
        "mean": embeddings.mean().item(),
        "std": embeddings.std().item(),
        "l2_norm_mean": embeddings.norm(dim=1).mean().item()
    }, embeddings


In [3]:
# =============================
# TOP-K RETRIEVAL (BY AUTHOR)
# =============================
# Purpose: Check if embeddings cluster messages from the same author.
# Method:
# 1. Compute cosine similarity between all embeddings
# 2. For each message, check if top-k nearest neighbors include the same author
# 3. Compute proportion (accuracy)

def evaluate_topk_author_accuracy(embeddings, dataloader, k=5):
    embeddings = normalize(embeddings, dim=1)
    sims = embeddings @ embeddings.T
    sims.fill_diagonal_(-1)

    authors = [item["author"] for item in dataloader.dataset]

    topk = sims.topk(k, dim=1).indices
    correct = 0

    for i, idxs in enumerate(topk):
        if authors[i] in [authors[j] for j in idxs]:
            correct += 1

    return correct / len(authors)


In [4]:
# =============================
# STYLE FEATURE CORRELATION
# =============================
# Purpose: Check if embeddings encode specific stylistic features
# Features: emoji count, uppercase ratio, punctuation ratio

def extract_style_features(texts):
    emoji_counts = [sum(1 for c in t if emoji.is_emoji(c)) for t in texts]
    uppercase_ratio = [sum(1 for c in t if c.isupper()) / max(len(t), 1) for t in texts]
    punct_ratio = [sum(1 for c in t if c in "!?.") / max(len(t), 1) for t in texts]
    return {
        "emoji": np.array(emoji_counts),
        "caps": np.array(uppercase_ratio),
        "punct": np.array(punct_ratio)
    }

def evaluate_style_correlations(embeddings, texts):
    sims = cosine_similarity(embeddings)
    style_feats = extract_style_features(texts)

    results = {}
    for name, feat in style_feats.items():
        diff = np.abs(feat[:, None] - feat[None, :])
        corr = np.corrcoef(sims.flatten(), -diff.flatten())[0, 1]
        results[name] = corr

    return results


In [5]:
# =============================
# CLUSTER VISUALIZATION (t-SNE)
# =============================
# Purpose: Visual check if messages with the same author or style form clusters
# Method: Reduce embeddings to 2D using t-SNE and plot

def save_tsne_plot(embeddings, authors, out_path):
    z2d = TSNE(n_components=2, random_state=42).fit_transform(embeddings.numpy())
    colors = [hash(a) % 100 for a in authors]

    plt.figure(figsize=(8,6))
    plt.scatter(z2d[:,0], z2d[:,1], c=colors, cmap="tab20", s=5)
    plt.title("t-SNE of Style Embeddings")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()


In [6]:
# =============================
# ABLATION TEST: Emoji removal
# =============================
# Purpose: Check if embeddings encode emojis
# Method: Remove emojis from content and compare embedding shift

# --------------------------------------------------
# Assumptions:
# - model is already loaded and trained
# - tokenizer is already loaded
# - val_loader is built from ConversationDataset
# - ConversationDataset returns:
#   input_ids, attention_mask, author, content
# --------------------------------------------------


def run_style_ablation(
    model, tokenizer, texts, ablation_fn, device, batch_size=8
):
    def embed(texts):
        zs = []
        for i in range(0, len(texts), batch_size):
            enc = tokenizer(
                texts[i:i+batch_size],
                padding=True,
                truncation=True,
                return_tensors="pt"
            )
            with torch.no_grad():
                z = model(
                    enc["input_ids"].to(device),
                    enc["attention_mask"].to(device)
                )
                zs.append(normalize(z, dim=1).cpu())
        return torch.cat(zs)

    z_orig = embed(texts)
    z_mod = embed([ablation_fn(t) for t in texts])

    shift = 1 - torch.nn.functional.cosine_similarity(z_orig, z_mod, dim=1)

    return {
        "mean": shift.mean().item(),
        "std": shift.std().item(),
        "min": shift.min().item(),
        "max": shift.max().item()
    }


In [7]:
def run_full_evaluation(
    model,
    tokenizer,
    val_loader,
    output_dir,
    device="cuda"
):
    os.makedirs(output_dir, exist_ok=True)
    model.to(device)

    texts = [item["content"] for item in val_loader.dataset]
    authors = [item["author"] for item in val_loader.dataset]

    results = {}

    # 1. Sanity
    sanity, embeddings = evaluate_embedding_sanity(model, val_loader, device)
    results["embedding_sanity"] = sanity

    # 2. Retrieval
    results["top5_author_accuracy"] = evaluate_topk_author_accuracy(
        embeddings, val_loader, k=5
    )

    # 3. Style correlation
    results["style_correlations"] = evaluate_style_correlations(
        embeddings.numpy(), texts
    )

    # 4. Ablations
    results["ablations"] = {
        "emoji": run_style_ablation(
            model, tokenizer, texts,
            lambda t: emoji.replace_emoji(t, ""),
            device
        ),
        "punctuation": run_style_ablation(
            model, tokenizer, texts,
            lambda t: t.translate(str.maketrans("", "", string.punctuation)),
            device
        ),
        "lowercase": run_style_ablation(
            model, tokenizer, texts,
            lambda t: t.lower(),
            device
        )
    }

    # 5. Save t-SNE
    save_tsne_plot(
        embeddings, authors,
        os.path.join(output_dir, "tsne.png")
    )

    # 6. Save JSON
    with open(os.path.join(output_dir, "results.json"), "w") as f:
        json.dump(results, f, indent=2)

    return results


In [8]:
model_name = "style_retriever_author_contrastive"
validation_data = "retriever_val_undersampled"

# Load model
save_dir = "../models/" + model_name

tokenizer = CanineTokenizer.from_pretrained(save_dir)
encoder = CanineModel.from_pretrained(save_dir)

model = StyleEncoder()
model.encoder = encoder
model.proj.load_state_dict(torch.load(f"{save_dir}/projection_head.pt"))
model.eval()
model.to("cuda")

# Load validation data
rows = pd.read_csv("../data/train/" + validation_data + ".csv").to_dict(orient="records")
val_dataset = ConversationDataset(rows, tokenizer, max_length=512)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

# Run evaluation
results = run_full_evaluation(
    model,
    tokenizer,
    val_loader,
    output_dir="../evaluations/" + model_name,
    device=device
)

In [9]:
model_name = "google/canine-s"  # base CANINE model
validation_data = "retriever_val_undersampled"


# Load tokenizer and encoder
tokenizer = CanineTokenizer.from_pretrained(model_name)
encoder = CanineModel.from_pretrained(model_name)

# Wrap in StyleEncoder
model = StyleEncoder()
model.encoder = encoder  # replace the encoder
model.to(device)
model.eval()

# Load validation data
rows = pd.read_csv(f"../data/train/{validation_data}.csv").to_dict(orient="records")
val_dataset = ConversationDataset(rows, tokenizer, max_length=512)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

# Run evaluation
results = run_full_evaluation(
    model,
    tokenizer,
    val_loader,
    output_dir=f"../evaluations/base_canine",
    device=device
)